# EDA-01 · Catalog Profile
Phân bổ category/segment, price tier, gross margin distribution.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10,
})

prods = pd.read_csv('products.csv')
prods['gross_margin_vnd'] = prods['price'] - prods['cogs']
prods['margin_pct']       = prods['gross_margin_vnd'] / prods['price'] * 100

# Price tiers (VND)
bins   = [0, 1_000, 3_000, 7_000, 15_000, 50_000]
labels = ['<1K', '1K–3K', '3K–7K', '7K–15K', '>15K']
prods['price_tier'] = pd.cut(prods['price'], bins=bins, labels=labels)

print(f'Products: {len(prods):,}  |  Categories: {prods["category"].nunique()}  |  Segments: {prods["segment"].nunique()}')
prods[['product_id','category','segment','price','cogs','gross_margin_vnd','margin_pct','price_tier']].head(5)

## 1. Category Distribution

In [ ]:
cat_count = prods['category'].value_counts().reset_index()
cat_count.columns = ['category','count']
cat_count['pct'] = cat_count['count'] / len(prods) * 100
print(cat_count.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# bar
axes[0].barh(cat_count['category'], cat_count['count'], color='steelblue')
axes[0].set_xlabel('Number of products')
axes[0].set_title('Product count by category')
for i, (c, p) in enumerate(zip(cat_count['count'], cat_count['pct'])):
    axes[0].text(c + 5, i, f'{c:,} ({p:.1f}%)', va='center', fontsize=9)

# pie
axes[1].pie(cat_count['count'], labels=cat_count['category'],
            autopct='%1.1f%%', startangle=140,
            colors=['#4C72B0','#DD8452','#55A868','#C44E52'])
axes[1].set_title('Category share')

plt.tight_layout()
plt.show()

## 2. Segment Distribution

In [ ]:
seg_count = prods['segment'].value_counts().reset_index()
seg_count.columns = ['segment','count']
seg_count['pct'] = seg_count['count'] / len(prods) * 100
print(seg_count.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(seg_count['segment'], seg_count['count'],
               color=plt.cm.tab10.colors[:len(seg_count)])
ax.set_xlabel('Number of products')
ax.set_title('Product count by segment')
for bar, (c, p) in zip(bars, zip(seg_count['count'], seg_count['pct'])):
    ax.text(bar.get_width() + 3, bar.get_y() + bar.get_height()/2,
            f'{c:,} ({p:.1f}%)', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 3. Category × Segment Cross-tab

In [ ]:
ct = pd.crosstab(prods['segment'], prods['category'])
print(ct)

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(ct.values, cmap='Blues', aspect='auto')
ax.set_xticks(range(len(ct.columns))); ax.set_xticklabels(ct.columns, rotation=20)
ax.set_yticks(range(len(ct.index)));   ax.set_yticklabels(ct.index)
ax.set_title('Product count: segment × category')
plt.colorbar(im, ax=ax, shrink=0.8)
for i in range(ct.shape[0]):
    for j in range(ct.shape[1]):
        v = ct.values[i, j]
        ax.text(j, i, str(v) if v > 0 else '', ha='center', va='center',
                fontsize=9, color='white' if v > ct.values.max()*0.6 else 'black')
plt.tight_layout()
plt.show()

## 4. Price Distribution

In [ ]:
print('Price (VND) summary:')
print(prods['price'].describe().apply(lambda x: f'{x:,.0f}').to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# histogram log scale
axes[0].hist(prods['price'], bins=50, color='steelblue', edgecolor='white', linewidth=0.4)
axes[0].set_xlabel('Price (VND)')
axes[0].set_ylabel('Count')
axes[0].set_title('Price distribution')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1000:.0f}K'))

# box by category
data   = [prods[prods['category']==c]['price'].values for c in prods['category'].unique()]
labels_cat = prods['category'].unique()
bp = axes[1].boxplot(data, labels=labels_cat, patch_artist=True,
                     medianprops={'color':'black','linewidth':1.5})
colors = ['#4C72B0','#DD8452','#55A868','#C44E52']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color); patch.set_alpha(0.7)
axes[1].set_ylabel('Price (VND)')
axes[1].set_title('Price by category (boxplot)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1000:.0f}K'))
plt.tight_layout()
plt.show()

## 5. Price Tier Analysis

In [ ]:
tier_count = prods['price_tier'].value_counts().sort_index().reset_index()
tier_count.columns = ['tier','count']
tier_count['pct'] = tier_count['count'] / len(prods) * 100

print('Price tier (VND):')
print(tier_count.to_string(index=False))

# By category
ct_tier = pd.crosstab(prods['price_tier'], prods['category'])
print('\nPrice tier × category:')
print(ct_tier)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# overall bar
axes[0].bar(tier_count['tier'].astype(str), tier_count['count'], color='#4C72B0')
axes[0].set_xlabel('Price tier (VND)')
axes[0].set_ylabel('Count')
axes[0].set_title('Products by price tier')
for i, (c, p) in enumerate(zip(tier_count['count'], tier_count['pct'])):
    axes[0].text(i, c + 3, f'{c}\n({p:.1f}%)', ha='center', fontsize=8)

# stacked by category
ct_tier.plot(kind='bar', stacked=True, ax=axes[1],
             color=['#4C72B0','#DD8452','#55A868','#C44E52'], edgecolor='white')
axes[1].set_xlabel('Price tier')
axes[1].set_ylabel('Count')
axes[1].set_title('Price tier × category (stacked)')
axes[1].legend(title='Category', bbox_to_anchor=(1, 1))
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

## 6. Gross Margin — Absolute (price − cogs, VND)

In [ ]:
print('Gross margin VND (price − cogs) summary:')
print(prods['gross_margin_vnd'].describe().apply(lambda x: f'{x:,.0f}').to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(prods['gross_margin_vnd'], bins=50, color='#55A868', edgecolor='white', linewidth=0.4)
axes[0].set_xlabel('Gross margin (VND)')
axes[0].set_ylabel('Count')
axes[0].set_title('Gross margin distribution (absolute VND)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1000:.0f}K'))

data_gm   = [prods[prods['category']==c]['gross_margin_vnd'].values for c in prods['category'].unique()]
bp2 = axes[1].boxplot(data_gm, labels=prods['category'].unique(), patch_artist=True,
                      medianprops={'color':'black','linewidth':1.5})
for patch, color in zip(bp2['boxes'], ['#4C72B0','#DD8452','#55A868','#C44E52']):
    patch.set_facecolor(color); patch.set_alpha(0.7)
axes[1].set_ylabel('Gross margin (VND)')
axes[1].set_title('Gross margin by category')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1000:.0f}K'))
plt.tight_layout()
plt.show()

## 7. Gross Margin % Distribution

In [ ]:
print('Gross margin % summary:')
print(prods['margin_pct'].describe().round(2).to_string())

# Margin buckets
mbins  = [0, 5.01, 10, 15, 20, 25, 30, 35, 40, 45, 50, 101]
mlbls  = ['≤5%','5-10%','10-15%','15-20%','20-25%','25-30%','30-35%','35-40%','40-45%','45-50%','50%+']
prods['margin_bucket'] = pd.cut(prods['margin_pct'], bins=mbins, labels=mlbls, include_lowest=True)
mb = prods['margin_bucket'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(mb.index.astype(str), mb.values, color='#C44E52', edgecolor='white', linewidth=0.4)
axes[0].set_xlabel('Margin bucket')
axes[0].set_ylabel('Count')
axes[0].set_title('Margin % distribution (all products)')
axes[0].tick_params(axis='x', rotation=45)

# by category
seg_margin = prods.groupby('category')['margin_pct'].mean().sort_values()
axes[1].barh(seg_margin.index, seg_margin.values, color='#C44E52', alpha=0.8)
axes[1].set_xlabel('Mean margin %')
axes[1].set_title('Average margin % by category')
for i, v in enumerate(seg_margin.values):
    axes[1].text(v + 0.2, i, f'{v:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 8. Margin % by Segment

In [ ]:
seg_stats = prods.groupby('segment')['margin_pct'].agg(['mean','median','std','min','max']).round(2)
seg_stats.columns = ['mean%','median%','std%','min%','max%']
seg_stats = seg_stats.sort_values('mean%', ascending=False)
print(seg_stats.to_string())

fig, ax = plt.subplots(figsize=(10, 5))
data_seg   = [prods[prods['segment']==s]['margin_pct'].values for s in seg_stats.index]
bp3 = ax.boxplot(data_seg, labels=seg_stats.index, patch_artist=True,
                 medianprops={'color':'black','linewidth':1.5}, vert=False)
cmap = plt.cm.RdYlGn
for i, patch in enumerate(bp3['boxes']):
    patch.set_facecolor(cmap(i / len(bp3['boxes']))); patch.set_alpha(0.75)
ax.set_xlabel('Gross margin %')
ax.set_title('Margin % distribution by segment')
ax.axvline(prods['margin_pct'].mean(), color='red', linestyle='--', alpha=0.6, label=f'Overall mean ({prods["margin_pct"].mean():.1f}%)')
ax.legend()
plt.tight_layout()
plt.show()

## 9. Price vs Margin % — Scatter by Category

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
color_map = {'Streetwear':'#4C72B0','Outdoor':'#DD8452','Casual':'#55A868','GenZ':'#C44E52'}
for cat, grp in prods.groupby('category'):
    ax.scatter(grp['price'], grp['margin_pct'],
               label=cat, alpha=0.4, s=15, color=color_map.get(cat,'gray'))
ax.set_xlabel('Price (VND)')
ax.set_ylabel('Gross margin %')
ax.set_title('Price vs Gross margin % by category')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1000:.0f}K'))
ax.legend(title='Category')
plt.tight_layout()
plt.show()

## 10. Price Tier × Margin % Summary

In [ ]:
tier_margin = prods.groupby('price_tier', observed=True)['margin_pct'].agg(
    count='count', mean='mean', median='median', std='std', min='min', max='max'
).round(2)
print('Margin % by price tier:')
print(tier_margin.to_string())

fig, ax = plt.subplots(figsize=(9, 4))
tm = prods.groupby('price_tier', observed=True)['margin_pct'].mean()
ax.bar(tm.index.astype(str), tm.values, color='#4C72B0', edgecolor='white')
ax.set_xlabel('Price tier (VND)')
ax.set_ylabel('Mean margin %')
ax.set_title('Average gross margin % by price tier')
for i, v in enumerate(tm.values):
    ax.text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## Summary

In [ ]:
print('=== Catalog Profile Summary ===')
print(f'Total SKUs       : {len(prods):,}')
print(f'Categories       : {prods["category"].nunique()}  ({", ".join(prods["category"].value_counts().index)})')
print(f'Segments         : {prods["segment"].nunique()}')
print()
print('Price (VND):')
for tier, cnt in prods["price_tier"].value_counts().sort_index().items():
    print(f'  {str(tier):>8}  {cnt:>4} SKUs ({cnt/len(prods)*100:.1f}%)')
print()
print(f'Gross margin %:')
print(f'  Overall mean   : {prods["margin_pct"].mean():.1f}%')
print(f'  Overall median : {prods["margin_pct"].median():.1f}%')
print(f'  Range          : {prods["margin_pct"].min():.1f}% – {prods["margin_pct"].max():.1f}%')
print()
print('By category:')
for cat, row in prods.groupby("category")["margin_pct"].agg(["mean","median"]).round(1).iterrows():
    print(f'  {cat:<12}  mean={row["mean"]}%  median={row["median"]}%')